# LLM Hallucination Guardrail — Analysis Notebook

End-to-end walkthrough: dataset → features → classifier → evaluation → live demo.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data.dataset import SAMPLES
print(f'Dataset size: {len(SAMPLES)} samples')
print(f'Hallucination rate: {sum(s["label"] for s in SAMPLES)/len(SAMPLES):.1%}')

## 1. Dataset Overview

In [ ]:
df = pd.DataFrame(SAMPLES)
print(df['domain'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['domain'].value_counts().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Samples per Domain')
axes[0].set_xlabel('')

df['label'].value_counts().plot.pie(
    ax=axes[1], labels=['Correct', 'Hallucination'],
    colors=['#4CAF50', '#f44336'], autopct='%1.0f%%'
)
axes[1].set_title('Label Distribution')

plt.tight_layout()
plt.show()

## 2. Feature Extraction (sample)

In [ ]:
from features.extractor import extract_features

# Compare a correct vs hallucinated answer
correct = SAMPLES[0]
wrong   = SAMPLES[1]

print('=== CORRECT ANSWER ===')
print(f'Q: {correct["question"]}')
print(f'A: {correct["answer"]}')
f_correct = extract_features(correct['question'], correct['answer'])
for k, v in f_correct.items():
    if not k.startswith('_'):
        print(f'  {k}: {v}')

print('\n=== HALLUCINATED ANSWER ===')
print(f'Q: {wrong["question"]}')
print(f'A: {wrong["answer"]}')
f_wrong = extract_features(wrong['question'], wrong['answer'])
for k, v in f_wrong.items():
    if not k.startswith('_'):
        print(f'  {k}: {v}')

## 3. Feature Matrix

In [ ]:
import json

# Load cached features if available (from train.py run)
cache_path = '../models/features_cache.json'
if os.path.exists(cache_path):
    with open(cache_path) as f:
        cache = json.load(f)
    X = np.array(cache['X'])
    y = np.array(cache['y'])
    print(f'Loaded cached features: {X.shape}')
else:
    print('Run python train.py first to cache features.')

feature_names = ['semantic_sim', 'confidence_score', 'ner_mismatch', 'contradiction_score', 'numeric_mismatch']
feat_df = pd.DataFrame(X, columns=feature_names)
feat_df['label'] = y
feat_df.groupby('label')[feature_names].mean().T.plot.bar(figsize=(10, 4), color=['#4CAF50', '#f44336'])
plt.title('Mean Feature Values: Correct (0) vs Hallucination (1)')
plt.legend(['Correct', 'Hallucination'])
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Precision-Recall Curve

In [ ]:
from IPython.display import Image
Image('../reports/pr_curve.png')

## 5. Model Metrics

In [ ]:
with open('../reports/metrics.json') as f:
    metrics = json.load(f)

pd.DataFrame(metrics).T[[
    'precision_hallucination', 'recall_hallucination',
    'f1_hallucination', 'average_precision', 'roc_auc'
]].rename(columns={
    'precision_hallucination': 'Precision',
    'recall_hallucination': 'Recall',
    'f1_hallucination': 'F1',
    'average_precision': 'AP',
    'roc_auc': 'ROC-AUC'
}).style.format('{:.2%}').background_gradient(cmap='RdYlGn')

## 6. Live API Demo

In [ ]:
import requests

test_cases = [
    {
        'question': 'What is the capital of France?',
        'answer': 'The capital of France is Paris.',
        'expected': 'CORRECT'
    },
    {
        'question': 'What is the capital of France?',
        'answer': 'The capital of France is definitely Lyon, the largest city.',
        'expected': 'HALLUCINATION'
    },
    {
        'question': 'How many bones are in the human body?',
        'answer': 'The adult human body has exactly 300 bones.',
        'expected': 'HALLUCINATION'
    },
]

print(f'{"Question":<45} {"Expected":<15} {"Risk Score":<12} {"Label":<8} {"Safe?"}')
print('-' * 95)

for tc in test_cases:
    try:
        r = requests.post('http://localhost:8000/score', json=tc, timeout=30)
        res = r.json()
        print(f"{tc['question'][:44]:<45} {tc['expected']:<15} {res['hallucination_risk']:<12} {res['risk_label']:<8} {res['safe_to_show']}")
    except Exception as e:
        print(f'  API not running: {e}')
        print('  Start API with: uvicorn api.main:app --reload')
        break